In [1]:
def print_text(text: str, n = 80) -> None:
    """
    Print the given text, inserting a newline character every 80 characters.

    :param text: The text to be printed.
    """
    
    for i in range(0, len(text), n):
        print(text[i:i+n])


def batch_embeddings(texts, batch_size=10):
    for i in range(0, len(texts), batch_size):
        response = openai.Embedding.create(input=texts[i:i + batch_size], engine="text-embedding-ada-002")
        embeddings = [item['embedding'] for item in response['data']]
        yield embeddings

In [3]:
import pandas as pd
import openai
import pinecone
from langchain.text_splitter import CharacterTextSplitter

PINECONE_API_KEY = "34db068a-4d0b-4c84-9ca5-3975f62478a4"
PINECONE_ENVIRONMENT = "gcp-starter"

index_name = "law-gpt"

# Initialize Pinecone
pinecone.init(api_key=PINECONE_API_KEY, environment=PINECONE_ENVIRONMENT)
if index_name not in pinecone.list_indexes():
    pinecone.create_index(index_name, dimension=1536)  # Ensure the dimension is correct
index = pinecone.Index(index_name)

df = pd.read_csv("processed_data.csv").head(200)


# Example usage
query_and_process_results("murder and divorce")


ID: 2776846-1, Score: 0.805666387
After the purchase of the pistol appellant went to the Luchi apartment, where, a
t about 6:30 in the evening, she, her husband and Luchi were joined by Mary Marr
ow. At the time Mary Marrow arrived at the apartment things appeared calm. Appel
lant was sitting on the couch watching television. Neal Gardner and Larry Luchi 
were, or had been, drinking. Mary Marrow washed some dishes then joined the othe
rs in the living room to watch television. She had been sitting in the living ro
om about ten minutes when she heard Neal Gardner mention to appellant something 
about leaving and getting a divorce. Appellant was heard to remark, “then you ar
e going to leave me.” To this Neal Gardner responded, “Yes.” A shot was heard. M
ary Marrow looked up and saw appellant standing over her husband holding a pisto
l. He was lying on the couch. Mary Marrow then heard and saw appellant fire anot
her shot.
###################################################
ID: 2776846-2

In [ ]:
import itertools
# Initialize the text splitter
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000)

def batch_embeddings(texts, batch_size=10):
    for i in range(0, len(texts), batch_size):
        response = openai.Embedding.create(input=texts[i:i + batch_size], engine="text-embedding-ada-002")
        embeddings = [item['embedding'] for item in response['data']]
        yield embeddings


# Initialize Pinecone
pinecone.init(api_key=PINECONE_API_KEY, environment=PINECONE_ENVIRONMENT)
if index_name not in pinecone.list_indexes():
    pinecone.create_index(index_name, dimension=1536)  # Ensure the dimension is correct
index = pinecone.Index(index_name)

# Prepare data with metadata
data_to_upload = []
for _, row in df.iterrows():
    # Split each row of text into chunks
    chunks = text_splitter.split_text(row['text'])

    # Generate embeddings for each chunk
    for chunk_index, chunk in enumerate(chunks):
        # Since we're processing one chunk at a time, wrap it in a list
        chunk_embedding = next(batch_embeddings([chunk]))

        # Metadata for each chunk
        metadata = {'text': chunk, 'original_id': row['id']}

        # Format each record as (id, vector, metadata)
        # Assuming each chunk_embedding is a list with a single embedding
        data_to_upload.append((f"{row['id']}-{chunk_index}", chunk_embedding[0], metadata))

# Helper function to break the data into batches and return as a list
def create_batches(data, batch_size=100):
    """Create and return a list of batch-sized chunks from data."""
    return [data[i:i + batch_size] for i in range(0, len(data), batch_size)]

# Define batch size
batch_size = 100  # Adjust this based on Pinecone's limitations and your requirements

# Create batches
batches = create_batches(data_to_upload, batch_size=batch_size)

# Upsert data in batches
for batch in batches:
    index.upsert(vectors=batch)
    print(f"Upserted a batch of size {len(batch)}")



In [ ]:
def query_and_process_results(query_text, min_text_length=30, top_k=15):
    """
    Query the Pinecone index with the given text and process the results.

    Args:
    query_text (str): The text to query.
    min_text_length (int): Minimum length of text to include in results.
    top_k (int): Number of top results to retrieve.

    Returns:
    None
    """
    # Generate the query vector
    query_vector = next(batch_embeddings([query_text]))[0]

    # Perform the query and request metadata
    query_results = index.query(vector=query_vector, top_k=top_k, include_metadata=True)

    # Process and print the results
    for result in query_results["matches"]:
        # Check if metadata and text are present and meet length requirement
        if 'metadata' in result and 'text' in result['metadata'] and len(result['metadata']['text']) > min_text_length:
            print(f"ID: {result['id']}, Score: {result['score']}")
            print_text(result['metadata']['text'])
            print("###################################################")

# Example usage
query_and_process_results("murder and divorce")


ID: 2776846-1, Score: 0.805666387
After the purchase of the pistol appellant went to the Luchi apartment, where, a
t about 6:30 in the evening, she, her husband and Luchi were joined by Mary Marr
ow. At the time Mary Marrow arrived at the apartment things appeared calm. Appel
lant was sitting on the couch watching television. Neal Gardner and Larry Luchi 
were, or had been, drinking. Mary Marrow washed some dishes then joined the othe
rs in the living room to watch television. She had been sitting in the living ro
om about ten minutes when she heard Neal Gardner mention to appellant something 
about leaving and getting a divorce. Appellant was heard to remark, “then you ar
e going to leave me.” To this Neal Gardner responded, “Yes.” A shot was heard. M
ary Marrow looked up and saw appellant standing over her husband holding a pisto
l. He was lying on the couch. Mary Marrow then heard and saw appellant fire anot
her shot.
###################################################
ID: 2776846-2